概念：torch的计算图是什么？
是图论中的图，是一个有向无环图，每个节点表示一个操作。

参数：requires_grad
requires_grad = True 把这个张量加入计算图，会记录向量的每个操作。
反向传播开始时，需要根据这个操作图（计算图）进行链式求导，还原到最开始的变量

计算Q的梯度时，由于Q是矩阵，torch.backward只能对标量求解，因此需要做处理

external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)
相当于给Q 2*1乘了个E 1*2 矩阵
得到L = external_grad[0]×Q[0] + external_grad[1]×Q[1] = 1×Q[0] + 1×Q[1] = Q[0] + Q[1]

取巧用法。。下次不要用backward算向量了，手动处理成标量，信息更明确

In [63]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = 3*a**3 - b**2

external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)
print(9*a**2 == a.grad)
print(-2*b == b.grad)
# a是具体的向量，因此a.grad是具体的梯度
print(a.grad)
print(b.grad)

tensor([True, True])
tensor([True, True])
tensor([36., 81.])
tensor([-12.,  -8.])


torch.autograd是用来算雅可比矩阵的.
grad和backward都是autograd提供的机制

In [65]:
import torch
import torch.autograd as ag


a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = 3*a**3 - b**2

# Q 是向量，ag.grad 需要指定 grad_outputs（即「把 Q 变成标量」的权重 v，和 backward(gradient=...) 一样）
v = torch.ones_like(Q)   # 等价于 external_grad = [1., 1.]
grad_a, grad_b = ag.grad(outputs=Q, inputs=(a, b), grad_outputs=v)
print("ag.grad(Q, (a,b), grad_outputs=v) 得到:")
print("  grad_a:", grad_a)
print("  grad_b:", grad_b)


ag.grad(Q, (a,b), grad_outputs=v) 得到:
  grad_a: tensor([36., 81.])
  grad_b: tensor([-12.,  -8.])
